# K-Nearest Neighbors Regressor – Solution

**Short name (GitHub):** `KNN_Regress`  
**Lab source:** Codecademy *K-Nearest Neighbor Regressor* (movies from-scratch + weighted average + `KNeighborsRegressor`)  
**Language:** Python (NumPy + pandas + Matplotlib + scikit-learn)

Worked answers. Compare with your `KNN_Regress_Practice_Skeleton.ipynb` cells.  
Companion files: `KNN_Regress_Cheatsheet.docx`, `KNN_Regress_Reusable_Template.ipynb`, `knn_regress_flowchart.png`, `KNN_Regress.py`.

### Learning objectives
- Reuse Euclidean distance in 2-D and *n*-D
- Min-max normalize so budget (dollars) does not drown year / runtime
- Predict a **real-valued** IMDb-style rating as the mean of the *k* nearest ratings
- Upgrade the mean to an **inverse-distance weighted** average
- Fit `KNeighborsRegressor` (`weights="uniform"` vs `"distance"`)
- Sweep *k* with MAE / RMSE / R² (bias–variance for a continuous target)
- Alternate implementations (broadcast NumPy, Manhattan, `NearestNeighbors`)
- Extra practice: house prices and used-car prices
- Monte-Carlo: *k*, rating noise, sample size, extra noise features
- Rewrite the same forecast for an analyst, a product exec, a viewer, a non-specialist

### Data files
- `data/knn_movies_ratings.csv` — 80 films (duration, year, budget, rating)
- `data/knn_houses.csv` — 220 sales (sqft, year_built, baths → price_k)
- `data/knn_usedcars.csv` — 180 listings (mileage_k, age, engine_l → price_k)

### Flowchart
Open `knn_regress_flowchart.png` while you work.

### Classifier vs regressor (one sentence)
Classifier: majority **vote**. Regressor: average (or weighted average) of neighbor **numbers**.


## Inline cheat-sheet (keep this cell visible)

See also **`KNN_Regress_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Euclidean | $d(a,b)=\sqrt{\sum_j (a_j-b_j)^2}$ |
| Manhattan | $d_1(a,b)=\sum_j \|a_j-b_j\|$ |
| Min-max | $x'=(x-x_{\min})/(x_{\max}-x_{\min})$ |
| Uniform | $\hat{y}=\frac{1}{k}\sum_{i\in N_k} r_i$ |
| Weighted | $\hat{y}=\dfrac{\sum_{i\in N_k} r_i/d_i}{\sum_{i\in N_k} 1/d_i}$ |
| Guard | if $d_i=0$, that neighbor *is* the query — return its rating |
| Overfit | $k$ too small → one outlier rating owns $\hat{y}$ |
| Underfit | $k$ too large → $\hat{y}\approx$ global mean |
| sklearn | `KNeighborsRegressor(n_neighbors=k, weights="distance")` |
| Metrics | MAE, RMSE, $R^2$ — **not** accuracy |
| Split | `train_test_split(..., test_size=0.25, random_state=7)` |
| Scale | fit `MinMaxScaler` on **train** only |
| Cold start | a query with no nearby rows has nothing useful to average |

**Flow:** features → scale → split → distance → $k$ neighbors → average / weight → sweep $k$ → simulate.


## 0. Packages


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("Libraries loaded")


## 1. Distance between points (2-D)


In [ ]:
star_wars = [125, 1977]
raiders = [115, 1981]
mean_girls = [97, 2004]

def distance_2d(movie1, movie2):
    length_difference = (movie1[0] - movie2[0]) ** 2
    year_difference = (movie1[1] - movie2[1]) ** 2
    return (length_difference + year_difference) ** 0.5

print(distance_2d(star_wars, raiders))
print(distance_2d(star_wars, mean_girls))
print("closer to Star Wars:",
      "Raiders" if distance_2d(star_wars, raiders) < distance_2d(star_wars, mean_girls) else "Mean Girls")


## 2. Distance in *n* dimensions


In [ ]:
star_wars_3 = [125, 1977, 11_000_000]
raiders_3 = [115, 1981, 18_000_000]
mean_girls_3 = [97, 2004, 17_000_000]

def distance(a, b):
    squared = 0.0
    for i in range(len(a)):
        squared += (a[i] - b[i]) ** 2
    return squared ** 0.5

print(distance(star_wars_3, raiders_3))
print(distance(star_wars_3, mean_girls_3))


### Task 2.2 — NumPy alternate


In [ ]:
def distance_np(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b))

print(distance_np(star_wars_3, raiders_3))
print("match loop?", np.isclose(distance(star_wars_3, raiders_3), distance_np(star_wars_3, raiders_3)))


## 3. Min-max normalization


In [ ]:
release_dates = [1897.0, 1998.0, 2000.0, 1948.0, 1962.0,
                 1950.0, 1975.0, 1960.0, 2017.0, 1937.0]

def min_max_normalize(lst):
    lo, hi = min(lst), max(lst)
    span = hi - lo if hi != lo else 1.0
    return [(x - lo) / span for x in lst]

scaled = min_max_normalize(release_dates)
print(scaled)
print("1897 ->", scaled[0], "(oldest in the list, so ~0)")


## 4. Movie data + uniform regressor


In [ ]:
movies = pd.read_csv("data/knn_movies_ratings.csv")
movie_dataset = {row.title: [row.duration, row.year, row.budget]
                 for row in movies.itertuples()}
movie_ratings = {row.title: row.rating for row in movies.itertuples()}
print("n movies:", len(movie_dataset))
print("Life of Pi features:", movie_dataset["Life of Pi"])
print("Life of Pi rating:", movie_ratings["Life of Pi"])


### Task 4.2 — column-wise min-max


In [ ]:
def fit_minmax(dataset):
    X = np.array(list(dataset.values()), dtype=float)
    return X.min(axis=0), X.max(axis=0)

def apply_minmax(point, mins, maxs):
    point = np.asarray(point, dtype=float)
    span = np.where(maxs - mins == 0, 1.0, maxs - mins)
    return ((point - mins) / span).tolist()

def normalize_dataset(dataset):
    mins, maxs = fit_minmax(dataset)
    return {t: apply_minmax(v, mins, maxs) for t, v in dataset.items()}, mins, maxs

movie_dataset_n, mins, maxs = normalize_dataset(movie_dataset)
print("mins:", mins)
print("maxs:", maxs)
print("Life of Pi scaled:", movie_dataset_n["Life of Pi"])


### Task 4.3 — uniform `predict`


In [ ]:
def predict(unknown, dataset, movie_ratings, k):
    distances = []
    for title, movie in dataset.items():
        distances.append([distance(movie, unknown), title])
    distances.sort()
    neighbors = distances[:k]
    total = 0.0
    for _, title in neighbors:
        total += movie_ratings[title]
    return total / len(neighbors)


### Task 4.4 — Incredibles 2, uniform *k* = 5


In [ ]:
inc_raw = [118, 2018, 200_000_000]
inc_n = apply_minmax(inc_raw, mins, maxs)
print("Incredibles 2 scaled:", inc_n)
yhat_u = predict(inc_n, movie_dataset_n, movie_ratings, 5)
print("uniform k=5 forecast:", round(yhat_u, 3))


## 5. Weighted regression


In [ ]:
def predict_weighted(unknown, dataset, movie_ratings, k, eps=1e-12):
    distances = []
    for title, movie in dataset.items():
        distances.append([distance(movie, unknown), title])
    distances.sort()
    neighbors = distances[:k]
    numerator = 0.0
    denominator = 0.0
    for dist, title in neighbors:
        d = max(dist, eps)
        numerator += movie_ratings[title] / d
        denominator += 1.0 / d
    return numerator / denominator

toy_r = [5.0, 6.8, 9.0]
toy_d = [3.2, 11.5, 1.1]
uni = sum(toy_r) / len(toy_r)
wavg = sum(r / d for r, d in zip(toy_r, toy_d)) / sum(1 / d for d in toy_d)
print(f"toy uniform={uni:.2f}  toy weighted={wavg:.2f}  (lesson: 6.93 vs ~7.90)")


### Task 5.2 — Incredibles 2 weighted


In [ ]:
yhat_w = predict_weighted(inc_n, movie_dataset_n, movie_ratings, 5)
print("weighted k=5 forecast:", round(yhat_w, 3))
print("delta vs uniform:", round(yhat_w - yhat_u, 3))


## 6. Alternate neighbor search


In [ ]:
titles = list(movie_dataset_n)
X = np.array([movie_dataset_n[t] for t in titles], dtype=float)
y = np.array([movie_ratings[t] for t in titles], dtype=float)

def predict_np(unknown, X, y, k, weighted=False, eps=1e-12):
    d = np.linalg.norm(X - np.asarray(unknown, float), axis=1)
    idx = np.argsort(d)[:k]
    if not weighted:
        return float(y[idx].mean())
    dd = np.maximum(d[idx], eps)
    w = 1.0 / dd
    return float(np.sum(y[idx] * w) / w.sum())

print("np uniform", round(predict_np(inc_n, X, y, 5, False), 3),
      "match?", np.isclose(predict_np(inc_n, X, y, 5, False), yhat_u))
print("np weighted", round(predict_np(inc_n, X, y, 5, True), 3),
      "match?", np.isclose(predict_np(inc_n, X, y, 5, True), yhat_w))


### Task 6.2 — Manhattan


In [ ]:
def predict_manhattan(unknown, X, y, k):
    d = np.abs(X - np.asarray(unknown, float)).sum(axis=1)
    idx = np.argsort(d)[:k]
    return float(y[idx].mean()), [titles[i] for i in idx]

man_y, man_nb = predict_manhattan(inc_n, X, y, 5)
print("Manhattan k=5:", round(man_y, 3), "neighbors:", man_nb)


### Task 6.3 — NearestNeighbors


In [ ]:
nn = NearestNeighbors(n_neighbors=5, metric="euclidean").fit(X)
dist, idx = nn.kneighbors([inc_n])
print("NN distances:", dist[0])
print("NN titles:", [titles[i] for i in idx[0]])
print("NN uniform mean:", y[idx[0]].mean())


## 7. scikit-learn `KNeighborsRegressor`


In [ ]:
regressor = KNeighborsRegressor(n_neighbors=5, weights="distance")
regressor.fit(X, y)

# invent a short cheap 2005 comedy
comedy_raw = [95, 2005, 8_000_000]
comedy_n = apply_minmax(comedy_raw, mins, maxs)
lop_n = movie_dataset_n["Life of Pi"]

queries = np.array([inc_n, comedy_n, lop_n], dtype=float)
print("queries (scaled):")
print(queries)
print("predictions:", regressor.predict(queries))
print("Life of Pi actual:", movie_ratings["Life of Pi"],
      "(in-sample, so the closest neighbor is itself)")


## 8. Sweep *k* and graph


In [ ]:
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.25, random_state=7)
k_list = list(range(1, 21))
mae_uniform, mae_distance = [], []
for k in k_list:
    pu = KNeighborsRegressor(n_neighbors=k, weights="uniform").fit(Xtr, ytr).predict(Xva)
    pw = KNeighborsRegressor(n_neighbors=k, weights="distance").fit(Xtr, ytr).predict(Xva)
    mae_uniform.append(mean_absolute_error(yva, pu))
    mae_distance.append(mean_absolute_error(yva, pw))

best_u = k_list[int(np.argmin(mae_uniform))]
best_w = k_list[int(np.argmin(mae_distance))]
print("best uniform k, MAE:", best_u, round(min(mae_uniform), 3))
print("best distance k, MAE:", best_w, round(min(mae_distance), 3))
print("k=5 uniform MAE:", round(mae_uniform[4], 3),
      "k=5 distance MAE:", round(mae_distance[4], 3))
print("R2 uniform best-k:",
      round(r2_score(yva, KNeighborsRegressor(n_neighbors=best_u).fit(Xtr, ytr).predict(Xva)), 3))


### Task 8.2 — plot


In [ ]:
plt.figure(figsize=(8, 4.6))
plt.plot(k_list, mae_uniform, marker="o", label="uniform")
plt.plot(k_list, mae_distance, marker="s", label="distance")
plt.axvline(best_u, ls="--", alpha=0.5)
plt.xlabel("k")
plt.ylabel("Validation MAE")
plt.title("Movie rating KNN")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 9. More practice


In [ ]:
houses = pd.read_csv("data/knn_houses.csv")
Xh = houses[["sqft", "year_built", "baths"]].to_numpy(float)
yh = houses["price_k"].to_numpy(float)
Xhtr, Xhva, yhtr, yhva = train_test_split(Xh, yh, test_size=0.2, random_state=11)
hsc = MinMaxScaler()
Xhtr_s, Xhva_s = hsc.fit_transform(Xhtr), hsc.transform(Xhva)

h_mae = []
for k in range(1, 31):
    p = KNeighborsRegressor(n_neighbors=k).fit(Xhtr_s, yhtr).predict(Xhva_s)
    h_mae.append(mean_absolute_error(yhva, p))
hk = int(np.argmin(h_mae)) + 1
print("houses n=", len(houses), "best k=", hk, "MAE=", round(min(h_mae), 2),
      "mean price_k=", round(yh.mean(), 1))

query_h = hsc.transform([[1800, 2005, 2.0]])
print("1800sqft / 2005 / 2 bath ŷ =",
      round(KNeighborsRegressor(n_neighbors=hk).fit(Xhtr_s, yhtr).predict(query_h)[0], 1), "kUSD")


### Task 9.2 — used cars


In [ ]:
cars = pd.read_csv("data/knn_usedcars.csv")
Xc = cars[["mileage_k", "age", "engine_l"]].to_numpy(float)
yc = cars["price_k"].to_numpy(float)
Xctr, Xcva, yctr, ycva = train_test_split(Xc, yc, test_size=0.2, random_state=3)
csc = MinMaxScaler()
Xctr_s, Xcva_s = csc.fit_transform(Xctr), csc.transform(Xcva)

c_mae = []
for k in range(1, 25):
    p = KNeighborsRegressor(n_neighbors=k).fit(Xctr_s, yctr).predict(Xcva_s)
    c_mae.append(mean_absolute_error(ycva, p))
ck = int(np.argmin(c_mae)) + 1
print("cars n=", len(cars), "best k=", ck, "MAE=", round(min(c_mae), 3),
      "mean price_k=", round(yc.mean(), 2))

query_c = csc.transform([[72.0, 7, 2.0]])
print("7yr / 72k miles / 2.0L ŷ =",
      round(KNeighborsRegressor(n_neighbors=ck).fit(Xctr_s, yctr).predict(query_c)[0], 2), "kUSD")


## 10. Simulation (edit the box, re-run)


In [ ]:
# ----- editable -----
K_FIXED = 5
NOISE = 0.00
TRAIN_FRAC = 0.75
N_EXTRA = 0
WEIGHTS = "uniform"
N_REPS = 12
SEED0 = 0
# --------------------

def one_run(seed):
    rng = np.random.default_rng(seed)
    XX = X.copy()
    y_tr_src = y + rng.normal(0, NOISE, size=y.shape)
    if N_EXTRA > 0:
        extra = rng.normal(0, 1, size=(len(XX), N_EXTRA))
        XX = np.hstack([XX, extra])
        mn, mx = XX.min(0), XX.max(0)
        XX = (XX - mn) / np.where(mx - mn == 0, 1.0, mx - mn)
    n_tr = max(K_FIXED + 2, int(TRAIN_FRAC * len(XX)))
    idx = rng.permutation(len(XX))
    tr, va = idx[:n_tr], idx[n_tr:]
    if len(va) < 5:
        va, tr = idx[-8:], idx[:-8]
    k = min(K_FIXED, len(tr))
    model = KNeighborsRegressor(n_neighbors=k, weights=WEIGHTS)
    model.fit(XX[tr], y_tr_src[tr])
    return mean_absolute_error(y[va], model.predict(XX[va]))

maes = [one_run(SEED0 + i) for i in range(N_REPS)]
print(f"mean MAE={np.mean(maes):.3f}  sd={np.std(maes):.3f}  "
      f"k={K_FIXED} noise={NOISE} frac={TRAIN_FRAC} extra={N_EXTRA} w={WEIGHTS}")

# small built-in panel so the solution notebook also shows the shape
grid_k = list(range(1, 16))
plt.figure(figsize=(7.2, 3.8))
plt.plot(grid_k, [np.mean([one_run(10 + i) if False else
                           mean_absolute_error(
                               y[va],
                               KNeighborsRegressor(n_neighbors=k).fit(X[tr], y[tr]).predict(X[va])
                           )
                           ]) for k in grid_k] if False else
         [mean_absolute_error(
             yva, KNeighborsRegressor(n_neighbors=k).fit(Xtr, ytr).predict(Xva)
         ) for k in grid_k])
plt.xlabel("k"); plt.ylabel("val MAE"); plt.title("re-used split from §8")
plt.grid(True, alpha=0.3)
plt.show()


## 11. Audience rewrite


In [ ]:
analyst = (
    "Scaled Euclidean KNN on 80 films × 3 features (duration, year, budget), "
    "75/25 split (random_state=7). Uniform k=5 forecasts Incredibles 2 at about 7.44; "
    "inverse-distance weighting lifts that to about 7.80 because closer prestige-era "
    "neighbors (Life of Pi, Inception, The Dark Knight) outvote the farther mid-rating titles. "
    "Hold-out MAE bottoms near 0.65 when k is large on this small table — the signal is weak "
    "once you leave the local pocket. Fit MinMax on train only; a 2018 year legitimately scales > 1."
)
product_exec = (
    "The recommender does not invent a rating. It looks up the five already-rated titles "
    "that look most like the new release on runtime, year and budget, then averages those "
    "scores. For Incredibles 2 that average sits a bit under 8 after we let nearer titles "
    "count more. On leftover films the typical miss is about two-thirds of a star. "
    "This is a cold-start helper for titles with no user history yet — not a substitute "
    "for actual reviews once they arrive."
)
viewer = (
    "To guess a star rating for a film you have not seen, we line up a handful of films "
    "that are a similar length, from a similar year, and made on a similar budget, then "
    "average their ratings. For Incredibles 2 that guess is a high 7. It will be wrong "
    "sometimes — usually by less than one star on films we held back as a test. "
    "It is a starting point, not a verdict on whether you will like the movie."
)
friend = (
    "Find a few movies that look like this one and average their scores. Closer matches "
    "get a louder vote. That is the whole method. Too few matches copies flukes; too many "
    "just repeats the catalogue average. Brand-new catalogues have the same problem as a "
    "brand-new user: nothing nearby to average yet."
)
for label, text in [("analyst", analyst), ("exec", product_exec),
                    ("viewer", viewer), ("nonspecialist", friend)]:
    print("====", label, "====")
    print(text)
    print()


## Done

You now have a uniform and a weighted from-scratch regressor, a sklearn check, two extra tables, and a knob box for *k* / noise / *n* / extra columns. Reuse `KNN_Regress_Reusable_Template.ipynb` on the next numeric target.
